# Aadhaar OCR Test

Extracts Aadhaar number, DOB, Gender, and Address from an Aadhaar card image using EasyOCR.

In [ ]:
import cv2
import easyocr
import re
import os

reader = easyocr.Reader(['en', 'hi'], gpu=False)

In [ ]:
def extract_aadhaar(image_path):
    if not os.path.exists(image_path):
        print(f"Error: Image file '{image_path}' not found!")
        return None

    print(f"Reading image: {image_path}")
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    print("Running text extraction...")
    results = reader.readtext(gray, detail=1)

    texts = [r[1] for r in results]
    full_text = " ".join(texts)

    print("\n--- Raw Text Extracted ---")
    for i, (bbox, text, conf) in enumerate(results):
        y = int(bbox[0][1])
        print(f"  [{i:>2}] [y={y:>4}] {text}")
    print("--------------------------\n")

    aadhaar_pattern = r'\b\d{4}\s?\d{4}\s?\d{4}\b'
    dob_pattern = r'(?:DOB|Date of Birth|Year of Birth)[:\s]*(\d{2}/\d{2}/\d{4}|\d{4})'
    gender_pattern = r'\b(Male|Female|MALE|FEMALE|TRANSGENDER)\b'
    name_pattern = r'Government of India[^A-Za-z]*([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)'
    father_pattern = r'Father[:\s]*([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)'

    aadhaar_matches = re.findall(aadhaar_pattern, full_text)
    aadhaar_number = aadhaar_matches[0].replace(' ', '') if aadhaar_matches else None

    # Card type: long cards have "Your Aadhaar No." label section
    aadhaar_label_idx = None
    for i, (bbox, text, _) in enumerate(results):
        if re.search(r'Aadhaar No', text):
            aadhaar_label_idx = i
            break

    card_type = "long" if aadhaar_label_idx is not None else "short"

    dob = re.search(dob_pattern, full_text, re.IGNORECASE)
    gender = re.search(gender_pattern, full_text, re.IGNORECASE)
    name = re.search(name_pattern, full_text)
    father = re.search(father_pattern, full_text, re.IGNORECASE)

    address = "Not Applicable"
    if aadhaar_label_idx is not None and aadhaar_label_idx > 4:
        addr_texts = []
        for i in range(4, aadhaar_label_idx):
            text = results[i][1].strip()
            if not text.isascii():
                continue
            if re.search(r'[A-Z]{2}\d{6,}', text):
                continue
            addr_texts.append(text)
        address = ", ".join(addr_texts) if addr_texts else "Not Found"

    result = {
        "status": "success",
        "card_type": card_type,
        "aadhaar_found": aadhaar_number is not None,
        "data": {
            "aadhaar_number": aadhaar_number,
            "name": name.group(1) if name else "Not Found",
            "father_name": father.group(1) if father else "Not Found",
            "dob": dob.group(1) if dob else "Not Found",
            "gender": gender.group(0).upper() if gender else "Not Found",
            "address": address
        }
    }

    print("Parsed JSON Result:")
    return result

In [ ]:
image_path = "s.jpeg"
extract_aadhaar(image_path)